<a href="https://colab.research.google.com/github/ole0246/LanguageModelsAsCognitiveModels/blob/main/DSC_291_HW1_Evaluating_BabyLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 1: Instructions, Setup, and Introduction to Minicons

In this homework, you will gain practice evaluating LMs using minimal pairs, and also make your own mini-research contributions.

1. **Section 1 (0 points):** Setup and intro to the minicons library for scoring sentence surprisal
2. **Section 2 (3 points):** BLiMP evaluation
3. **Section 3 (4 points):** Write your own minimal pairs datasets
4. **Section 4 (5 points):** Mini-project
5. **Section 5 (0 points):** Submit

Note: We've decided to split HW1 up into two parts. Part 1 is worth 12 points. Part 2 will be released later and is worth 13 points.

**GENERATIVE AI POLICY:**
For this homework, the use of LLMs is allowed for brainstorming and code generation BUT you need to do two things:

- You must disclose all generative AI use.
- You must describe all generative AI contributions *in your own words*.

## Instructions, Setup, and Introduction to Minicons (0 Points)

- Step 1: Make a copy of this notebook.
- Step 2: Connect to GPU (if needed).
   - We recommend connecting to T4 GPUs for compute-intensive tasks, like evaluating models on lots of minimal pairs. To do so, go to `Runtime > Change runtime type > Hardware accelerator` and select `T4 GPU`. There are usage limits (for free accounts), so you may want to connect to GPU only when needed. This notebook can also be run on university clusters where free GPUs are provided (let us know if you need support with this).
- Step 3: Run the setup cells.
- Step 4: Complete the assignment.
- Step 5: Convert the notebook to .pdf (see instructions at the bottom).
- Step 6: Submit using Gradescope (link on the course canvas site).

In [ ]:
# Install required packages
!pip install git+https://github.com/kanishkamisra/minicons.git
!pip install datasets
!pip install "transformers<4.50" -q

In [ ]:
from minicons import scorer
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from datasets import load_dataset

## Minicons

[Minicons](https://github.com/kanishkamisra/minicons) is a Python library that provides a simple interface for computing token-level and sentence-level surprisal scores from pretrained language models. It supports both **decoder-only** (causal/incremental) models like GPT-2 and **encoder-only** (masked) models like BERT.

**What does Minicons do?**

- Computes **surprisal** (negative log-probability) at the token and sequence level.
- Supports multiple scoring strategies: raw log-probability, normalized surprisal, etc.
- Works with any HuggingFace model out of the box. *NOTE: Currently, support for MLMs only seems to be compatible with Transformers versions 4.50 or earlier.*

**Why is this useful?**

Surprisal-based evaluation allows us to do **zero-shot grammaticality judgments** — we can compare the surprisal of a grammatical sentence to an ungrammatical one without any task-specific fine-tuning. A well-trained language model should assign **lower surprisal** (higher probability) to the grammatical sentence.

**Scorer types:**

- `scorer.IncrementalLMScorer` — for decoder-only / causal LMs (e.g., GPT-2, DistilGPT-2)
- `scorer.MaskedLMScorer` — for encoder-only / masked LMs (e.g., BERT)

**Reduction functions for `sequence_score`:**

- Sequence Surprisal: `lambda x: -x.sum(0).item()`
- Normalized Surprisal: `lambda x: -x.mean(0).item()`
- Sequence Log-probability: `lambda x: x.sum(0).item()`
- Normalized Log-probability: `lambda x: x.mean(0).item()`

Now we can use Minicons with Decoder-Only and Encoder-Only Models

Below we demonstrate how to use `minicons` with both types of models on example minimal pairs. **Minimal pairs** are pairs of sentences that differ in exactly one grammatical feature, allowing us to isolate what a model knows about that specific phenomenon.

In [ ]:
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

# Load models
ilm_model = scorer.IncrementalLMScorer('gpt2-large', device) # Autoregressive (Decoder) LM
mlm_model = scorer.MaskedLMScorer('bert-base-uncased', device)  # Bidirectional (Masked) LM

# Example minimal pairs
stimuli = [
    ["The keys to the cabinet are on the table.", "The keys to the cabinet is on the table."],
    ["Buddy is easy to sleep.", "Buddy is eager to sleep."],
    ["It's herself that Sharon talked about.", "It's herself that talked about Sharon."],
    ["There seems to be a dog under the table.", "There tried to be a dog under the table."],
    ["Buddy dropped the bone.", "Buddy fell the bone."],
    ["I know that you saw the dog.", "I know who you saw the dog."],
]

# Score and display
for i, stim in tqdm(enumerate(stimuli, 1)):
    print(f"--- Stimulus {i} ---")
    print(f"Sentences: {stim}")
    scores_ilm = ilm_model.sequence_score(stim, reduction=lambda x: -x.sum(0).item())
    scores_mlm = mlm_model.sequence_score(stim, reduction=lambda x: -x.sum(0).item())

    print(f"GPT2-large (autoregressive) surprisal: {scores_ilm}")
    print(f"BERT (bidirectional) surprisal: {scores_mlm}")
    print()

# Section 2: BLiMP Evaluation (3 points)

[BLiMP](https://github.com/alexwarstadt/blimp) (Benchmark of Linguistic Minimal Pairs) consists of 67 sub-datasets, each containing 1,000 minimal pairs that isolate specific contrasts in syntax, morphology, or semantics.

**Task:**

1. Pick **three phenomena** (sub-tasks) from BLiMP that interest you. Try to pick tasks that **vary in difficulty**.
2. Write a **function** to evaluate a model on BLiMP.
3. Evaluate Pythia checkpoints on BLiMP sub-tasks.
  - Tip: You can limit evaluation to 100 minimal pairs per sub-task, rather than 1000.
4. **Plot** the results as a line plots comparing the change in performance throughout training.
5. **Explain** your results: Which model performs better? Why might that be?

## Select BLiMP Phenomena

You can browse the available sub-tasks and model performance in Table 4 [here](https://arxiv.org/pdf/1912.00582). Make sure to pick sub-tasks of different difficulty. Some examples:

- Easy
    - `anaphor_gender_agreement`
    - `existential_there_subject_raising`
    - `determiner_noun_agreement_with_adj_2`
    - `irregular_past_participle_adjectives`
- Medium
    - `causative`
    - `ellipsis_n_bar_2`
    - `wh_island`
    - `superlative_quantifiers_1`
- Hard
    - `principle_A_c_command`
    - `wh_vs_that_with_gap`
    - `matrix_question_npi_licensor_present`
    - `distractor_agreement_relative_clause`

In [ ]:
# Run this block to list all BLiMP subtasks by their official task_names
from datasets import get_dataset_config_names

# Get all sub-task names (configurations) for BLiMP
blimp_tasks = get_dataset_config_names("nyu-mll/blimp")

print(f"Total BLiMP sub-tasks: {len(blimp_tasks)}")
print("\n".join(blimp_tasks))

Your BLiMP Sub-Tasks

1. \<YOUR ANSWER HERE\>
2. \<YOUR ANSWER HERE\>
3. \<YOUR ANSWER HERE\>

## Complete the evaluation function

In [ ]:
# Complete this evaluation function. Add any helper functions you need.

def evaluate(task_name, ms_scorer, normalize=False, n=100):
    """Evaluate a model on a BLiMP sub-task.
       - task_name: a BLiMP task name
       - ms_scorer: a minicons scorer
       - normalize: whether or not to normalize surprisal by number of tokens
       - n: number of minimal pairs to evaluate on
       - Returns: Accuracy (0 to 1)."""
    ds = load_dataset("nyu-mll/blimp", task_name)

    reduction = (lambda x: -x.mean(0).item()) if normalize else (lambda x: -x.sum(0).item())

    # Use select to get a subset as a dataset object which preserves the dictionary structure per row
    subset = ds['train'].select(range(min(len(ds['train']), n)))

    correct = 0
    total = 0
    for ex in tqdm(subset, desc=f"Evaluating on {task_name}", leave=False):
        pass  # <YOUR CODE HERE>
    # return accuracy



## Evaluate Pythia Checkpoints

[Pythia](https://huggingface.co/EleutherAI/pythia-160m-deduped-v0) is a family of open-source and open-weight LLMs trained by the non-profit Eleuther AI.

To aid interpretability research, Eleuther released checkpoints throughout training for all the Pythia models, meaning we can track how model performance changes throughout pretraining.

You will evaluate 4 checkpoints at roughly log-linear intervals.



In [ ]:
from transformers import GPTNeoXForCausalLM, AutoTokenizer

def load_pythia(ckpt):
    model = GPTNeoXForCausalLM.from_pretrained(
    "EleutherAI/pythia-70m-deduped",
    revision=f"step{ckpt}",
    cache_dir=f"./pythia-70m-deduped/step{ckpt}",
    )

    tokenizer = AutoTokenizer.from_pretrained(
    "EleutherAI/pythia-70m-deduped",
    revision=f"step{ckpt}",
    cache_dir=f"./pythia-70m-deduped/step{ckpt}",
    )
    return model, tokenizer

steps = [128, 1000, 10000, 143000]
ckpts = {}
for step in steps:
    ckpts[step] = load_pythia(step)

In [ ]:
# Evaluate your models and put the results into a pandas dataframe

tasks = [] # <YOUR TASKS>
for ckpt in ckpts.keys():
    ms_scorer = scorer.IncrementalLMScorer(ckpts[ckpt][0], device, tokenizer=ckpts[ckpt][1])
    for task in blimp_tasks:
        pass  # <YOUR CODE HERE>

# Package your outputs into a pandas dataframe called `results_df`
# print(results_df.head())
# print(results_df.describe())

## Plot the results

In [ ]:
# Plot your results in a line plot using seaborn

# <YOUR CODE HERE>

## Generative AI Usage
Describe any contributions made by generative AI in this portion of the assignment.

*[Your answer here]*

# Section 3: Create Your Own Evaluation Pairs (4 points)

**Mini-Benchmark:** Create **at least 3** minimal pairs datasets of your own with at least 5 pairs each. For each dataset:

1. Select a phenomenon to test. In a markdown cell, explain **what phenomenon** you are testing (e.g., subject-verb agreement, reflexive binding, negative polarity items, etc.).
2. Write 5 pairs with one good sentence and its bad counterpart.
3. Score your pairs using `minicons` with the Pythia checkpoints. Plot the results as before.
4. Summarize your findings.

Some ideas for minimal pair datasets are provided in this [google doc](https://docs.google.com/document/d/1jRyKssx0haSfnh4g94YNI4C5mKvGgWzWM3af7Acz7_8/edit?tab=t.0). You are welcome to use these ideas or come up with your own.

## Describe the phenomena you are testing with each pair

\<YOUR ANSWER HERE\>

## Your mini-benchmark

In [ ]:
# Your minimal pairs datasets
dataset1 = [
    {"sentence_good": "",  # TODO
     "sentence_bad": ""},  # TODO
    # x10
]

dataset2 = [
    {"sentence_good": "",  # TODO
     "sentence_bad": ""},  # TODO
    # x10
]

dataset3 = [
    {"sentence_good": "",  # TODO
     "sentence_bad": ""},  # TODO
    # x10
]

## Evaluate Pythia on your mini-benchmark

In [ ]:
# Evaluate the models and plot your results

#<YOUR CODE HERE>

## Your Analysis
Summarize your findings. Did the models behave as expected? Why or why not?

*[Your answer here]*

## Generative AI Usage
Describe any contributions made by generative AI in this portion of the assignment.

*[Your answer here]*

# Section 4 (Mini-Project): Extend Your Work (5 points)

This is your first mini-project. We invite you to extend this type of work in some way that interests you. Some sample ideas are provided below, but as before, these are just suggestions.

*The use of generative AI is permitted and even encouraged for this part. If code generation can help you accomplish more, that is a great outcome!*

**Extra credit may be given to innovative or high-quality contributions!**

Suggested topics:

1. Extend your minimal pair mini-benchmark.
2. Evaluate LMs on a different minimal pair dataset.
3. Study the effect of model size (e.g., compare different GPT2 or Pythia sizes) on performance on minimal pair datasets.
4. Compare consistency across models on minimal pair datasets. Do different models find the same exampels easy/hard?
5. Browse and evaluate other models and model families on [huggingface](https://huggingface.co/models) and from the [BabyLM challenge](https://huggingface.co/babylm/models).

## Your Code

In [ ]:
# <YOUR CODE HERE>

## Describe your mini-project
You may write-up your motivation, methods, and results however you like. Small text blocks can be interleaved with code blocks, or you can write up everything in one text block.

## Generative AI Usage
Describe any contributions made by generative AI in this portion of the assignment.

*[Your answer here]*

# Section 5: Convert to .pdf

This section will walk you through instructions to convert your notebook to .pdf.

- Step 0: Ensure your notebook's outputs are displayed properly.
  - Make sure all cells were run in order and their outputs are as intended.
  - Clear any outputs from cells that don't need to be assessed.
- Step 1: Download your notebook as an `.ipynb` file.
  - Go to `File > Download > Download .ipynb`.
- Step 2: Drag and drop your `.ipynb` file into the `contents` directory of your notebook. You can find this by clicking on the folder icon on the left-hand side of the colab notebook. Ensure the filename in the `!jupyter nbconvert` command matches the uploaded file.
- Step 3: Run the codeblock below.
- Step 4: Download the output .pdf from the `content` directory. It should look like a nicely formatted LaTeX document.
  - Common error: In markdown files, ensure that all bulleted or numbered lists have an empty line above them (unless they are the first line in the markdown file). Otherwise, they will not render properly and **we may deduct points**.
- Step 5: Upload your assignment to gradescope.


In [ ]:
# Install the necessary LaTeX packages for PDF conversion. (This will take a minute or two.)
!apt-get install texlive-xetex texlive-fonts-recommended texlive-plain-generic

# Convert your notebook to PDF. Make sure the notebook name is correct.
!jupyter nbconvert --to pdf /content/DSC_291_HW1_Evaluating_BabyLMs.ipynb

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
texlive-fonts-recommended is already the newest version (2021.20220204-1).
texlive-plain-generic is already the newest version (2021.20220204-1).
texlive-xetex is already the newest version (2021.20220204-1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
[NbConvertApp] Converting notebook /content/DSC_291_HW1_Evaluating_BabyLMs.ipynb to pdf
[NbConvertApp] Writing 50041 bytes to notebook.tex
[NbConvertApp] Building PDF
[NbConvertApp] Running xelatex 3 times: ['xelatex', 'notebook.tex', '-quiet']
[NbConvertApp] Running bibtex 1 time: ['bibtex', 'notebook']
[NbConvertApp] WARNING | bibtex had problems, most likely because there were no citations
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 81672 bytes to /content/DSC_291_HW1_Evaluating_BabyLMs.pdf
